# C3 — C Tuning Standalone

Notebook riêng để chạy `StratifiedKFold` CV chọn `C` cho SVM classical và QSVM statevector. Kết quả được cache theo `CONFIG_TAG` để chạy lại không recompute.

In [52]:
import os, json, warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms import QSVC

import qiskit, qiskit_machine_learning
print(f'Qiskit    : {qiskit.__version__}')
print(f'Qiskit ML : {qiskit_machine_learning.__version__}')

Qiskit    : 2.3.0
Qiskit ML : 0.9.0


In [53]:
# ── Cấu hình giống ─────────────────────────────────────────
RANDOM_STATE = 42
N_QUBITS     = 4
ANGLE_MAX    = np.pi

REPS         = 2
ENTANGLEMENT = 'full'
C_QSVM       = 3.0
SHOTS        = 'sv'

REPS_Z       = 2
C_QSVM_Z     = 3.0

SVM_MODEL_KEYS = ['linear_mm', 'linear_std', 'poly_mm', 'poly_std', 'rbf_mm', 'rbf_std']
SVM_C_VALUES   = np.array([0.1, 0.1, 0.1, 0.1, 0.1, 10.0], dtype=float)
SVM_C_BY_MODEL = dict(zip(SVM_MODEL_KEYS, SVM_C_VALUES))

def _fmt_c_for_tag(c):
    return f'{float(c):g}'.replace('.', 'p')

SVM_C_TAG = '-'.join(
    f'{key.replace("_", "")}{_fmt_c_for_tag(SVM_C_BY_MODEL[key])}'
    for key in SVM_MODEL_KEYS
)

POLY_DEGREE  = 2
RBF_GAMMA    = 'scale'

LABEL_COLS   = ['label', 'label_binary', 'label_multiclass', 'attack_category']

DATA_DIR       = '../data/processed_data'
MODELS_DIR     = '../models'
QSVM_MODEL_DIR = f'{MODELS_DIR}/qsvm_cache'
os.makedirs(QSVM_MODEL_DIR, exist_ok=True)

TRAIN_SIZE     = 1000
TEST_SIZE      = 300
KM_SAMPLE_SIZE = 100


CONFIG_TAG = (
    f'r{REPS}_{ENTANGLEMENT}'
    f'_cq{C_QSVM}'
    f'_cs{SVM_C_TAG}'
    f'_s{SHOTS}'
    f'_n{TRAIN_SIZE}'
    f'_t{TEST_SIZE}'
    f'_km{KM_SAMPLE_SIZE}'
)

RESULTS_CACHE_DIR = f'{QSVM_MODEL_DIR}/results_c3kg_{CONFIG_TAG}'
os.makedirs(RESULTS_CACHE_DIR, exist_ok=True)

np.random.seed(RANDOM_STATE)
print(f'CONFIG_TAG        : {CONFIG_TAG}')
print(f'RESULTS_CACHE_DIR : {RESULTS_CACHE_DIR}')

CONFIG_TAG        : r2_full_cq3.0_cslinearmm0p1-linearstd0p1-polymm0p1-polystd0p1-rbfmm0p1-rbfstd10_ssv_n1000_t300_km100
RESULTS_CACHE_DIR : ../models/qsvm_cache/results_c3kg_r2_full_cq3.0_cslinearmm0p1-linearstd0p1-polymm0p1-polystd0p1-rbfmm0p1-rbfstd10_ssv_n1000_t300_km100


In [54]:
# ── Load transformers đã fit trên full train — KHÔNG fit lại ─────────────
selector = joblib.load(f'{MODELS_DIR}/feature_selector_k20.joblib')
pca      = joblib.load(f'{MODELS_DIR}/pca_4components.joblib')
scaler   = joblib.load(f'{MODELS_DIR}/scaler_minmax_pi.joblib')

print(f'SelectKBest : k={selector.k}, score_func={selector.score_func.__name__}')
print(f'PCA         : n_components={pca.n_components_}, variance={pca.explained_variance_ratio_.sum()*100:.2f}%')
print(f'Scaler      : range=[{scaler.feature_range[0]:.4f}, {scaler.feature_range[1]:.4f}]')

def transform_pipeline(df, feature_cols):
    """Cleaned CSV -> angle-encoded numpy array [0, pi]."""
    X_raw = df[feature_cols].to_numpy(dtype=np.float32)
    X_sel = selector.transform(X_raw)
    X_pca = pca.transform(X_sel)
    X_ang = np.clip(scaler.transform(X_pca), 0, ANGLE_MAX)
    return X_ang

def pca_only(df, feature_cols):
    """Raw -> PCA output trước MinMax scaler, dùng cho StandardScaler branch."""
    X_raw = df[feature_cols].to_numpy(dtype=np.float32)
    X_sel = selector.transform(X_raw)
    return pca.transform(X_sel)

SelectKBest : k=20, score_func=f_classif
PCA         : n_components=4, variance=86.62%
Scaler      : range=[0.0000, 3.1416]


In [55]:
# ── Load train set ────────────────────────────────────────────────────────
train_path = f'{DATA_DIR}/NSL_KDD_Train_Sample{TRAIN_SIZE}.csv'

if not os.path.exists(train_path):
    raise FileNotFoundError(f'[MISSING] {train_path}')

df_train     = pd.read_csv(train_path)
feature_cols = [c for c in df_train.columns if c not in LABEL_COLS]

# MinMax [0, pi] branch: dùng cho QSVM và *_mm classical.
X_train = transform_pipeline(df_train, feature_cols)
y_train = df_train['label_binary'].to_numpy(dtype=np.int64)

# StandardScaler branch: khớp C3 chính, fit trên PCA output của train.
X_train_pca = pca_only(df_train, feature_cols)
std_scaler  = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train_pca)

print(f'Train path  : {train_path}')
print(f'X_train     : {X_train.shape} | class dist={np.bincount(y_train)}')
print(f'X_train_std : {X_train_std.shape} | range=[{X_train_std.min():.3f}, {X_train_std.max():.3f}]')
print(f'feature_cols: {len(feature_cols)}')

Train path  : ../data/processed_data/NSL_KDD_Train_Sample1000.csv
X_train     : (1000, 4) | class dist=[506 494]
X_train_std : (1000, 4) | range=[-2.750, 2.838]
feature_cols: 122


In [56]:
def build_quantum_kernel():
    """FidelityStatevectorKernel với ZZFeatureMap, deterministic/noiseless."""
    fm = zz_feature_map(
        feature_dimension=N_QUBITS,
        reps=REPS,
        entanglement=ENTANGLEMENT,
    )
    return FidelityStatevectorKernel(
        feature_map=fm,
        shots=None,
        enforce_psd=True,
        cache_size=None,
    )

In [57]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  C TUNING — StratifiedKFold CV trên X_train                            ║
# ║  Mục đích: đảm bảo fairness trong comparison, KHÔNG phải fix overfit   ║
# ║  Kết quả cache theo CONFIG_TAG → không recompute khi chạy lại           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Toggle & config ───────────────────────────────────────────────────────
RUN_C_TUNING      = True
C_TUNING_CV_FOLDS = 5
C_CANDIDATES      = [0.1, 0.3, 0.5, 1.0, 3.0, 5.0, 10.0]
C_TUNING_SCORING  = 'f1_macro'

_c_tune_cache_path = f'{RESULTS_CACHE_DIR}/c_tuning_{CONFIG_TAG}.json'

def _select_best_C(rows):
    """
    True 1-SE rule: chọn C nhỏ nhất có mean >= best_mean - SE(best).

    Với SVM, C nhỏ hơn tương ứng regularization mạnh hơn / model đơn giản hơn.
    Lưu ý: SE = std across folds / sqrt(n_folds), không phải std.
    """
    best_row = max(rows, key=lambda r: r['mean'])
    best_se = best_row.get('std_err', best_row['std'] / np.sqrt(C_TUNING_CV_FOLDS))
    threshold = best_row['mean'] - best_se
    eligible = [r for r in rows if r['mean'] >= threshold]
    chosen = min(eligible, key=lambda r: r['C'])
    return chosen['C']

def _tune_C_classical(kernel_kwargs, X_tr, y_tr):
    skf  = StratifiedKFold(n_splits=C_TUNING_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for c in C_CANDIDATES:
        svc    = SVC(C=c, probability=True, random_state=RANDOM_STATE, **kernel_kwargs)
        scores = cross_val_score(svc, X_tr, y_tr, cv=skf, scoring=C_TUNING_SCORING)
        row = {
            'C': float(c),
            'mean': float(scores.mean()),
            'std': float(scores.std(ddof=1)),
            'std_err': float(scores.std(ddof=1) / np.sqrt(len(scores))),
            'n_folds': int(len(scores)),
        }
        rows.append(row)
        print(f'    C={c:<5}  cv_f1={row["mean"]:.4f} ± {row["std"]:.4f} '
              f'(SE={row["std_err"]:.4f})')
    return _select_best_C(rows), rows

def _tune_C_qsvm(X_tr, y_tr):
    skf  = StratifiedKFold(n_splits=C_TUNING_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for c in C_CANDIDATES:
        fold_scores = []
        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            qk   = build_quantum_kernel()
            qsvc = QSVC(quantum_kernel=qk, C=c, random_state=RANDOM_STATE)
            qsvc.fit(X_tr[tr_idx], y_tr[tr_idx])
            fold_scores.append(f1_score(y_tr[val_idx], qsvc.predict(X_tr[val_idx]), average='macro'))
        fold_scores = np.asarray(fold_scores, dtype=float)
        row = {
            'C': float(c),
            'mean': float(fold_scores.mean()),
            'std': float(fold_scores.std(ddof=1)),
            'std_err': float(fold_scores.std(ddof=1) / np.sqrt(len(fold_scores))),
            'n_folds': int(len(fold_scores)),
        }
        rows.append(row)
        print(f'    C={c:<5}  cv_f1={row["mean"]:.4f} ± {row["std"]:.4f} '
              f'(SE={row["std_err"]:.4f})')
    return _select_best_C(rows), rows

c_tune_results = {}

if not RUN_C_TUNING:
    print('[C TUNING] Skipped.')

elif os.path.exists(_c_tune_cache_path):
    with open(_c_tune_cache_path) as fp:
        c_tune_results = json.load(fp)
    print(f'[C TUNING] Loaded từ cache → {_c_tune_cache_path}')

else:
    print(f'[C TUNING] {C_TUNING_CV_FOLDS}-fold CV | candidates={C_CANDIDATES} | scoring={C_TUNING_SCORING}\n')

    classical_configs = [
        ('linear_mm',  dict(kernel='linear'),                                  X_train),
        ('linear_std', dict(kernel='linear'),                                  X_train_std),
        ('poly_mm',    dict(kernel='poly', degree=POLY_DEGREE, gamma='scale'), X_train),
        ('poly_std',   dict(kernel='poly', degree=POLY_DEGREE, gamma='scale'), X_train_std),
        ('rbf_mm',     dict(kernel='rbf', gamma=RBF_GAMMA),                   X_train),
        ('rbf_std',    dict(kernel='rbf', gamma=RBF_GAMMA),                   X_train_std),
    ]

    for key, kw, X_tr in classical_configs:
        print(f'  [classical] {key} ...')
        best_C, rows = _tune_C_classical(kw, X_tr, y_train)
        c_tune_results[key] = {'best_C': float(best_C), 'cv_rows': rows}
        print(f'    → best C = {best_C}\n')

    print('  [QSVM ZZFeatureMap] ...')
    best_C_q, rows_q = _tune_C_qsvm(X_train, y_train)
    c_tune_results['qsvm'] = {'best_C': float(best_C_q), 'cv_rows': rows_q}
    print(f'    → best C = {best_C_q}\n')

    with open(_c_tune_cache_path, 'w') as fp:
        json.dump(c_tune_results, fp, indent=2)
    print(f'[C TUNING] Saved → {_c_tune_cache_path}')


[C TUNING] 5-fold CV | candidates=[0.1, 0.3, 0.5, 1.0, 3.0, 5.0, 10.0] | scoring=f1_macro

  [classical] linear_mm ...
    C=0.1    cv_f1=0.8530 ± 0.0248 (SE=0.0111)
    C=0.3    cv_f1=0.8502 ± 0.0248 (SE=0.0111)
    C=0.5    cv_f1=0.8493 ± 0.0235 (SE=0.0105)
    C=1.0    cv_f1=0.8493 ± 0.0235 (SE=0.0105)
    C=3.0    cv_f1=0.8502 ± 0.0216 (SE=0.0097)
    C=5.0    cv_f1=0.8502 ± 0.0216 (SE=0.0097)
    C=10.0   cv_f1=0.8502 ± 0.0216 (SE=0.0097)
    → best C = 0.1

  [classical] linear_std ...
    C=0.1    cv_f1=0.8500 ± 0.0253 (SE=0.0113)
    C=0.3    cv_f1=0.8492 ± 0.0234 (SE=0.0105)
    C=0.5    cv_f1=0.8493 ± 0.0235 (SE=0.0105)
    C=1.0    cv_f1=0.8493 ± 0.0235 (SE=0.0105)
    C=3.0    cv_f1=0.8502 ± 0.0216 (SE=0.0097)
    C=5.0    cv_f1=0.8502 ± 0.0216 (SE=0.0097)
    C=10.0   cv_f1=0.8502 ± 0.0216 (SE=0.0097)
    → best C = 0.1

  [classical] poly_mm ...
    C=0.1    cv_f1=0.8657 ± 0.0190 (SE=0.0085)
    C=0.3    cv_f1=0.8600 ± 0.0201 (SE=0.0090)
    C=0.5    cv_f1=0.8568 ± 0.0177

In [58]:
# ── Bảng kết quả + gợi ý cập nhật config ─────────────────────────────────
if c_tune_results:
    print()
    print(f'=== C TUNING RESULTS ({C_TUNING_CV_FOLDS}-fold CV, true 1-SE rule, scoring={C_TUNING_SCORING}) ===')
    print(f'  {"Model":>14}  {"Chosen C":>8}  {"CV F1":>8}  {"std":>7}  {"SE":>7}')
    print('  ' + '-' * 57)
    for key, val in c_tune_results.items():
        best_C = val['best_C']
        chosen_row = next(r for r in val['cv_rows'] if np.isclose(float(r['C']), float(best_C)))
        std_err = chosen_row.get('std_err', chosen_row['std'] / np.sqrt(C_TUNING_CV_FOLDS))
        print(f'  {key:>14}  {best_C:>8}  {chosen_row["mean"]:>8.4f}  '
              f'{chosen_row["std"]:>7.4f}  {std_err:>7.4f}')

    print()
    print('=== GỢI Ý CẬP NHẬT CONFIG TRONG C3 ===')
    if 'qsvm' in c_tune_results:
        cq = c_tune_results['qsvm']['best_C']
        print(f'  C_QSVM   = {cq}')
        print(f'  C_QSVM_Z = {cq}   # giữ bằng để fair ablation ZZ vs Z')

    tuned_c_values = [c_tune_results[key]['best_C'] for key in SVM_MODEL_KEYS if key in c_tune_results]
    if len(tuned_c_values) == len(SVM_MODEL_KEYS):
        print(f'  SVM_MODEL_KEYS = {SVM_MODEL_KEYS}')
        print(f'  SVM_C_VALUES   = np.array({tuned_c_values}, dtype=float)')

    print(f'\nCache file: {_c_tune_cache_path}')



=== C TUNING RESULTS (5-fold CV, true 1-SE rule, scoring=f1_macro) ===
           Model  Chosen C     CV F1      std       SE
  ---------------------------------------------------------
       linear_mm       0.1    0.8530   0.0248   0.0111
      linear_std       0.1    0.8500   0.0253   0.0113
         poly_mm       0.1    0.8657   0.0190   0.0085
        poly_std       0.1    0.8589   0.0167   0.0075
          rbf_mm       0.1    0.8632   0.0137   0.0061
         rbf_std      10.0    0.9099   0.0118   0.0053
            qsvm       1.0    0.9038   0.0266   0.0119

=== GỢI Ý CẬP NHẬT CONFIG TRONG C3 ===
  C_QSVM   = 1.0
  C_QSVM_Z = 1.0   # giữ bằng để fair ablation ZZ vs Z
  SVM_MODEL_KEYS = ['linear_mm', 'linear_std', 'poly_mm', 'poly_std', 'rbf_mm', 'rbf_std']
  SVM_C_VALUES   = np.array([0.1, 0.1, 0.1, 0.1, 0.1, 10.0], dtype=float)

Cache file: ../models/qsvm_cache/results_c3kg_r2_full_cq3.0_cslinearmm0p1-linearstd0p1-polymm0p1-polystd0p1-rbfmm0p1-rbfstd10_ssv_n1000_t300_km100/c_t